# Fig. 2: Effective model

Squeezing spectrum of the effective model, in which the cooling atoms and the auxiliary mode have been adiabatically eliminated: the squeezed cavity mode $a_s = a\cosh r + a^\dagger\sinh r$ decays at rate $\kappa_f$ (`dec_f`) with frequency shift $\Delta$ (`delta_lamb_f`), on top of the bare cavity decay $\kappa$. Same $(r, N_c)$ grid as `1_full_model_cumulants.ipynb`.

**Outputs** (`../data/`): `Ssq_min_eff.npy`, `Ssq_cut_eff.npy` (panel c). The axes ($N_c$, $r$, $\nu$) are saved by notebook 1.

In [ ]:
using QuantumCumulants
using ModelingToolkit
using OrdinaryDiffEq
using WignerSymbols
using ProgressMeter
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)

    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_r = 150 # Squeezing parameter r
num_N = 150 # Number of cooling atoms N_c

ind_r_cut = 100 # Index of r (r ≈ 8.7 dB) at which the squeezing spectrum S(ν) is saved (top of panel c)

# Rb87 level structure (B in G; angular frequencies, with the frequency in GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380 # D1 line
w_t_D2 = 2*pi*384230.48446 # D2 line
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

N_atom = logrange(100,2e4,num_N) # Number of cooling atoms N_c

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = LinRange(0.01,1.5,num_r) # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s).+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and the effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$. Arrays have shape `(num_r, num_N)`.

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav.+3*w_r.-w_s).-sqrt.(4*gr^2*N_atom'.+(w_cav.-w_r).^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom'.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom'*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)'./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

# Step 6: eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom'.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # Coefficient of the s's term

# Step 7: eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom').^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom')./((delta_lamb.^2).+(N_atom'.*dec/2).^2)

## Effective model

In [ ]:
@rnumbers Δ κ κ_f R # Parameters

h = FockSpace(:cavity)

a = Destroy(h,:a)
a_s = a*cosh(R)+a'*sinh(R) # Squeezed mode

H = Δ*a_s'*a_s

D = [a;a_s] # Collapse operators
rates = [κ; κ_f]

In [ ]:
ops = [a, a*a, a'a] # Operators whose equations we derive
eqs = meanfield(ops,H,D;rates=rates,order=2)
me_comp = complete!(eqs)

# Two-time correlation functions for the squeezing spectrum
corr = CorrelationFunction(a',a,eqs; steady_state=true)
corr2 = CorrelationFunction(a,a,eqs; steady_state=true)

@named sys = System(me_comp)

In [ ]:
param_list = (Δ, κ, κ_f, R)
S1 = Spectrum(corr,param_list)
S2 = Spectrum(corr2,param_list)

## Squeezing spectrum on the $(r, N_c)$ grid

Same procedure as in notebook 1: steady state at the largest $N_c$, then a sweep in $N_c$ re-using the previous solution.

In [ ]:
out = zeros(num_r,num_N) # Minimum of the squeezing spectrum

freqs = LinRange(-1,1,1000)*(2pi*1e-3)
spec = zeros(num_r,num_N,length(freqs)) # Squeezing spectrum

it = zeros(num_r,num_N)
dist = zeros(num_r,num_N)

progress = Progress(num_r*num_N)
sim_lock = ReentrantLock()

Threads.@threads for ind_r in 1:num_r

    # Initial steady state at the largest N_c
    u0 = zeros(ComplexF64, length(me_comp))

    ind_N = num_N

    p0 = (delta_lamb_f[ind_r,ind_N], kappa, dec_f[ind_r,ind_N], r[ind_r])
    dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

    prob = ODEProblem(sys,dict,(0.0,100000.0))
    sol = solve(prob, Tsit5(); abstol=1e-6, reltol=1e-6, maxiters=1e6)

    u0 = sol.u[end]

    # Sweep over N_c
    for ind_N in 1:num_N

        dist[ind_r,ind_N] = 10

        while dist[ind_r,ind_N]>1 && it[ind_r,ind_N]<15

            p0 = (delta_lamb_f[ind_r,ind_N], kappa, dec_f[ind_r,ind_N], r[ind_r])
            dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

            prob = ODEProblem(sys,dict,(0.0,10000.0))
            sol = solve(prob, Tsit5(); abstol=1e-6, reltol=1e-6, maxiters=1e5)

            lock(sim_lock) do
                dist[ind_r,ind_N] = maximum(abs.(sol[a'a]))-minimum(abs.(sol[a'a]))
            end

            u0 = sol.u[end]

            it[ind_r,ind_N] = it[ind_r,ind_N]+1

        end

        n_t = length(sol.t)
        u_avg = sum(sol.u)/n_t # Time-averaged state

        spec[ind_r,ind_N,:] = 1 .+ kappa*(S1(freqs,u_avg,p0) .+ S2(freqs,u_avg,p0) .+ S1(-freqs,u_avg,p0) .+ S2(-freqs,u_avg,p0))

        lock(sim_lock) do
            out[ind_r,ind_N] = minimum(spec[ind_r,ind_N,:])
        end

        next!(progress)
    end
end

## Spectra in dB

In [ ]:
# Unphysical negative values of the spectrum are set to 1 (0 dB) before taking the logarithm
out_clip = ifelse.(out .< 0, 1, out)
spec_clip = ifelse.(spec .< 0, 1, spec)

# Stored as -10 log10(S): positive values mean squeezing
Ssq_min = -20*log10.(sqrt.(out_clip))
Ssq_cut = -20*log10.(sqrt.(spec_clip[ind_r_cut,:,:]))

## Save

In [ ]:
data_dir = joinpath("..", "data") # Relative to this notebook

npzwrite(joinpath(data_dir, "Ssq_min_eff.npy"), Ssq_min)
npzwrite(joinpath(data_dir, "Ssq_cut_eff.npy"), Ssq_cut)